# 字节串(`bytes`)的构造与转换

In [1]:
import struct, io; 

In [2]:
#显示bytes对象的十六进制内容
def bytes_hexview(b: bytes): 
    strm_bin = io.BytesIO(b); 
    strm_hex = io.StringIO(b.hex()); 
    #每行显示8个字节
    while True: 
        #显示下一个字节相对于开头的偏移量
        print("{0:0>16x}".format(strm_bin.tell()), end="\x20" * 2); 
        ch_byte = strm_bin.read(8); 
        len_pedding = 8 - len(ch_byte); 
        #打印每个字节的十六进制码, 相邻字节间以空格分割
        for _ in range(8): 
            print(strm_hex.read(2), end="\x20"); 
        print("\x20" * (len_pedding * 2 + 1), end=str())
        #输出8个字节对应的字符
        #其中, 0x20至0x7e输出为对应的ASCII可打印字符, 其他字节输出为半角句点
        for ch in ch_byte: 
            if 32 <= ch <= 126: 
                print(chr(ch), end=str()); 
            else: 
                print(".", end=str()); 
        print("\n", end=str()); 
        if len_pedding > 0: 
            break; 

## `bytes`对象的显式构造
`bytes`是一种与`str`相似的对象, 在数据结构中同属于"串". 但`bytes`中每个元素都是字长固定为8位 (1字节), 取值介于`0`(含)~`255`(含)之间的整数. 

`bytes`的声明方法与`str`相似, 使用一对**半角**双引号(或一对**半角**单引号)括注若干字符, 但**需在定界符之前插入`b`**. 此外, `bytes`的定界符之内接受的字符类型存在限制, 只能包含以下字符: 

* 使用转义规则`\x`后接两位十六进制数码所表示的字符; 
* `\a`, `\b`, `\t`, `\n`, `\v`, `\f`, `\r` (即`\x07`至`\x0`); 
* `ASCII`可打印字符 (即`\x20`至`\x7e`), 但`"`, `'`和`\`需分别转义为`\"`, `\'`, `\\` (与`str`中的转义规则相同). 

In [3]:
motto = u"Verba volant, scripta manent"; 
motto_bin = b"Verba volant, scripta manent"; 
print(type(motto), type(motto_bin)); 

<class 'str'> <class 'bytes'>


## `bytes`对象的隐式构造

`bytes`对象的隐式构造方式包括: 
* `bytes`对象的拼接; 
* 采用特定编码规则, 对`str`对象中的字符内容进行编码; 
* 使用其他迭代器构造, 要求其中的每个元素均为`int`; 
* 使用其他迭代器构造, 要求其中的每个元素均为**字长固定, 类型相同**的其他对象, 如`array.array`, `np.ndarray`等; 
* 使用`struct.pack`方法构造

## `bytes`对象的拼接
支持`__add__`, `__mul__`和`join`方法, 功能和用法与`str`对象对应的同名方法相同, 但要求参与拼接的所有序列均为`bytes`, 不支持`str`与`bytes`的混合拼接. 

In [4]:
b"A" + b"B" * 3 + b"C"

b'ABBBC'

In [5]:
b" < ".join([b"1", b"2", b"3"])

b'1 < 2 < 3'

## `str`对象的编码与`bytes`对象的解码

|方法|功能|备注|
|:-|:-:|:-|
|`text.encode(rule)`|采用`rule`对`text`逐字符编码, <br>得到按字符顺序无重复无间隔<br>排列的二进制序列|`text`为`str`对象, <br>方法的返回结果为`bytes`对象|
|`binary.decode(rule)`|采用`rule`对`binary`解码|`binary`为`bytes`对象, <br>方法的返回结果为`str`对象|

对比基本拉丁字母和汉字分别在`UTF-8`, `UTF-16`和`UTF-32`编码下的差异

In [6]:
enc_rules = tuple("utf-{len:d}".format(len=l) for l in (8, 16, 32)); 

In [7]:
for rule in enc_rules: 
    print(rule); 
    bytes_hexview(motto.encode(rule)); 

utf-8
0000000000000000  56 65 72 62 61 20 76 6f  Verba vo
0000000000000008  6c 61 6e 74 2c 20 73 63  lant, sc
0000000000000010  72 69 70 74 61 20 6d 61  ripta ma
0000000000000018  6e 65 6e 74              nent
utf-16
0000000000000000  ff fe 56 00 65 00 72 00  ..V.e.r.
0000000000000008  62 00 61 00 20 00 76 00  b.a. .v.
0000000000000010  6f 00 6c 00 61 00 6e 00  o.l.a.n.
0000000000000018  74 00 2c 00 20 00 73 00  t.,. .s.
0000000000000020  63 00 72 00 69 00 70 00  c.r.i.p.
0000000000000028  74 00 61 00 20 00 6d 00  t.a. .m.
0000000000000030  61 00 6e 00 65 00 6e 00  a.n.e.n.
0000000000000038  74 00                    t.
utf-32
0000000000000000  ff fe 00 00 56 00 00 00  ....V...
0000000000000008  65 00 00 00 72 00 00 00  e...r...
0000000000000010  62 00 00 00 61 00 00 00  b...a...
0000000000000018  20 00 00 00 76 00 00 00   ...v...
0000000000000020  6f 00 00 00 6c 00 00 00  o...l...
0000000000000028  61 00 00 00 6e 00 00 00  a...n...
0000000000000030  74 00 00 00 2c 00 00 00  t...,...
00

In [8]:
for rule in enc_rules: 
    print(rule); 
    bytes_hexview("岁月失语，唯石能言".encode(rule)); 

utf-8
0000000000000000  e5 b2 81 e6 9c 88 e5 a4  ........
0000000000000008  b1 e8 af ad ef bc 8c e5  ........
0000000000000010  94 af e7 9f b3 e8 83 bd  ........
0000000000000018  e8 a8 80                 ...
utf-16
0000000000000000  ff fe 81 5c 08 67 31 59  ...\.g1Y
0000000000000008  ed 8b 0c ff 2f 55 f3 77  ..../U.w
0000000000000010  fd 80 00 8a              ....
utf-32
0000000000000000  ff fe 00 00 81 5c 00 00  .....\..
0000000000000008  08 67 00 00 31 59 00 00  .g..1Y..
0000000000000010  ed 8b 00 00 0c ff 00 00  ........
0000000000000018  2f 55 00 00 f3 77 00 00  /U...w..
0000000000000020  fd 80 00 00 00 8a 00 00  ........
0000000000000028                           
